# Session 10 Activity 1: RAGAS Evaluation with Cost Analysis

Compare an open-source RAG (Fireworks: gpt-oss-20b + Qwen3-embedding-8b) against a
closed RAG (OpenAI: gpt-4.1-mini + text-embedding-3-small) over the same cat-health
guide. Score both with RAGAS, capture token usage, and turn tokens into real dollars.

Run this in STAGES. Do not run the whole notebook blind:
1. Stage 1: cells up to and including the smoke test. Confirm both pipelines answer.
2. Stage 2: fill in the golden set, run both pipelines over it.
3. Stage 3: RAGAS eval + cost table.

Prereqs in your .env:
- FIREWORKS_API_KEY (already set)
- OPENAI_API_KEY (needed for gpt-4.1-mini, OpenAI embeddings, and the RAGAS judge)
- LANGCHAIN_API_KEY (for LangSmith tracing)


## Stage 1 — Setup

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

for key in ["FIREWORKS_API_KEY", "OPENAI_API_KEY", "LANGCHAIN_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(f"Enter {key}: ")

# LangSmith tracing on. Token counts get captured per run in this project.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "s10-oss-vs-closed-rag"
print("Env loaded. LangSmith project:", os.environ["LANGCHAIN_PROJECT"])

### Published pricing (per 1M tokens, USD, verified June 2026)

LangSmith cannot price the Fireworks model, so we apply these rates by hand to the
token counts we capture. Update these if rates change.


In [ ]:
# USD per 1,000,000 tokens
PRICING = {
    "fireworks_llm":   {"in": 0.07, "out": 0.30},   # gpt-oss-20b
    "openai_llm":      {"in": 0.40, "out": 1.60},   # gpt-4.1-mini
    "fireworks_embed": {"in": 0.008},               # qwen3-embedding-8b (approx; billed on input only)
    "openai_embed":    {"in": 0.02},                # text-embedding-3-small (input only)
}
def cost_usd(tokens, rate_per_million):
    return tokens / 1_000_000 * rate_per_million
print("Pricing loaded.")

### Load the cat guide once, chunk once (shared by both pipelines)

In [ ]:
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

RAG_DATA_DIR = os.environ.get("RAG_DATA_DIR", "data")

def tiktoken_len(text):
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))

loader = DirectoryLoader(RAG_DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=tiktoken_len
)
chunks = splitter.split_documents(documents)
print(f"Loaded {len(documents)} docs, split into {len(chunks)} chunks.")

### Pipeline A — Fireworks (open source)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate

FIREWORKS_BASE = "https://api.fireworks.ai/inference/v1"

RAG_PROMPT = ChatPromptTemplate.from_messages([("human",
    "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
    "Use the provided context to answer the query. "
    "Only use the provided context. If the answer is not in the context, "
    'respond with "I don\'t know".')])

# Fireworks embeddings (Qwen3-8b, 4096 dims)
fw_embed = OpenAIEmbeddings(
    model="accounts/fireworks/models/qwen3-embedding-8b",
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE,
    check_embedding_ctx_length=False,
    dimensions=4096,
)
fw_store = QdrantVectorStore.from_documents(
    documents=chunks, embedding=fw_embed,
    location=":memory:", collection_name="fw_collection",
)
fw_retriever = fw_store.as_retriever(search_kwargs={"k": 4})

fw_llm = ChatOpenAI(
    model="accounts/fireworks/models/gpt-oss-20b",
    temperature=0,
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE,
)
print("Fireworks pipeline ready.")

### Pipeline B — OpenAI (closed source)

In [ ]:
oa_embed = OpenAIEmbeddings(model="text-embedding-3-small")  # uses OPENAI_API_KEY
oa_store = QdrantVectorStore.from_documents(
    documents=chunks, embedding=oa_embed,
    location=":memory:", collection_name="oa_collection",
)
oa_retriever = oa_store.as_retriever(search_kwargs={"k": 4})

oa_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)  # uses OPENAI_API_KEY
print("OpenAI pipeline ready.")

### A RAG runner that returns the answer, the retrieved contexts, AND token usage

We call the LLM directly (not through a string parser) so we can read
`usage_metadata` off the response. That is how we get real token counts for costing.


In [ ]:
def run_rag(question, retriever, llm):
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    msg = RAG_PROMPT.format_messages(query=question, context=context)
    resp = llm.invoke(msg)
    usage = resp.usage_metadata or {}
    return {
        "answer": resp.content,
        "contexts": [d.page_content for d in docs],
        "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
    }
print("Runner defined.")

### Smoke test — one question through both pipelines (Stage 1 stop point)

In [ ]:
q = "What are the recommended core vaccinations for kittens?"

fw = run_rag(q, fw_retriever, fw_llm)
oa = run_rag(q, oa_retriever, oa_llm)

print("=== FIREWORKS (gpt-oss-20b) ===")
print(fw["answer"][:600])
print("tokens in/out:", fw["input_tokens"], fw["output_tokens"])
print("\n=== OPENAI (gpt-4.1-mini) ===")
print(oa["answer"][:600])
print("tokens in/out:", oa["input_tokens"], oa["output_tokens"])

## Stage 2 — Golden test set

FILL THIS IN from your cat guide. Write 5 to 6 questions the guide actually answers,
and a short reference answer for each in your own words from the document. RAGAS uses
`reference` for the retrieval metrics (context recall/precision). Do not fabricate;
pull each reference straight from the guide.


In [ ]:
golden_set = [
    {
        "question": "What are the core vaccinations recommended for cats?",
        "reference": "The core vaccines are rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV). FeLV vaccination is also considered core for kittens and young cats.",
    },
    {
        "question": "At what age range is a cat considered a mature adult?",
        "reference": "A mature adult cat is aged 7 to 10 years.",
    },
    {
        "question": "How should vaccines be administered to reduce the risk from injection-site sarcoma?",
        "reference": "Vaccines should be given in the lower limbs or tail, below the elbow or stifle, or in the distal third of the tail, so that surgical excision or amputation is possible if a sarcoma forms. The intrascapular space is not recommended.",
    },
    {
        "question": "What is the rule of thumb for how many litter boxes to provide in a multi-cat household?",
        "reference": "One litter box per cat plus one additional box, or one box per social group plus one additional box.",
    },
    {
        "question": "What daily energy requirement factor is used for senior cats over 10 years of age?",
        "reference": "For senior cats over 10 years, the resting energy requirement is multiplied by an added 10 to 20 percent, and in some cases up to 25 percent.",
    },
    {
        "question": "What surgical anesthesia protocol does the guide recommend for declawing?",
        "reference": "I don't know",
    },
]

### Run both pipelines over the golden set, collect records + token totals

In [ ]:
def collect(retriever, llm, label):
    records, tin, tout = [], 0, 0
    for item in golden_set:
        r = run_rag(item["question"], retriever, llm)
        records.append({
            "user_input": item["question"],
            "response": r["answer"],
            "retrieved_contexts": r["contexts"],
            "reference": item["reference"],
        })
        tin += r["input_tokens"]
        tout += r["output_tokens"]
        print(f"[{label}] {item['question'][:50]}... in={r['input_tokens']} out={r['output_tokens']}")
    return records, tin, tout

fw_records, fw_tin, fw_tout = collect(fw_retriever, fw_llm, "FW")
oa_records, oa_tin, oa_tout = collect(oa_retriever, oa_llm, "OA")
print("\nFireworks total tokens in/out:", fw_tin, fw_tout)
print("OpenAI    total tokens in/out:", oa_tin, oa_tout)

### Eyeball the "gap" question

The last golden-set item asks something the guide does not cover, so a well-behaved
pipeline should answer "I don't know" rather than make something up. Print both answers
side by side to see which one abstains.

In [ ]:
gap_index = len(golden_set) - 1
print("QUESTION:", golden_set[gap_index]["question"], "\n")
print("=== FIREWORKS answer ===")
print(fw_records[gap_index]["response"], "\n")
print("=== OPENAI answer ===")
print(oa_records[gap_index]["response"])

## Stage 3 — RAGAS evaluation

Judge model is gpt-4.1-mini (provider-neutral to the pipeline being scored). Metrics:
1. faithfulness: is the answer grounded in the retrieved context (no hallucination).
2. answer relevancy: does the answer actually address the question.
3. context precision: are the retrieved chunks relevant (retrieval quality).
4. context recall: did retrieval pull everything the reference needs (retrieval coverage).


In [ ]:
# ragas 0.4.3 still imports langchain_community.chat_models.vertexai, which newer
# langchain-community dropped. Point that name at langchain-google-vertexai (already
# installed) before ragas loads below. Nothing here uses Vertex; this just unblocks the import.
import sys, types

try:
    import langchain_community.chat_models.vertexai
except ModuleNotFoundError:
    from langchain_google_vertexai import ChatVertexAI
    shim = types.ModuleType("langchain_community.chat_models.vertexai")
    shim.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = shim

print("Ready to import ragas.")

In [ ]:
import asyncio
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0))
judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

metrics = {
    "faithfulness": Faithfulness(llm=judge_llm),
    "answer_relevancy": ResponseRelevancy(llm=judge_llm, embeddings=judge_emb),
    "context_precision": LLMContextPrecisionWithReference(llm=judge_llm),
    "context_recall": LLMContextRecall(llm=judge_llm),
}

async def score_records(records, label):
    per_metric = {name: [] for name in metrics}
    for i, rec in enumerate(records):
        sample = SingleTurnSample(
            user_input=rec["user_input"],
            response=rec["response"],
            retrieved_contexts=rec["retrieved_contexts"],
            reference=rec["reference"],
        )
        for name, metric in metrics.items():
            try:
                s = await metric.single_turn_ascore(sample)
            except Exception as e:
                s = float("nan")
                print(f"[{label}] row {i} {name} failed: {e}")
            per_metric[name].append(s)
        print(f"[{label}] scored row {i+1}/{len(records)}")
    # average each metric, ignoring nan
    import math
    averaged = {}
    for name, vals in per_metric.items():
        clean = [v for v in vals if not (isinstance(v, float) and math.isnan(v))]
        averaged[name] = sum(clean) / len(clean) if clean else float("nan")
    return averaged, per_metric

fw_scores, fw_detail = await score_records(fw_records, "FW")
oa_scores, oa_detail = await score_records(oa_records, "OA")

print("\n=== FIREWORKS (gpt-oss-20b) ===")
for k, v in fw_scores.items():
    print(f"{k}: {v:.3f}")
print("\n=== OPENAI (gpt-4.1-mini) ===")
for k, v in oa_scores.items():
    print(f"{k}: {v:.3f}")

### Cost computation from captured tokens

In [ ]:
def total_cost(tin, tout, llm_key):
    c_in = cost_usd(tin, PRICING[llm_key]["in"])
    c_out = cost_usd(tout, PRICING[llm_key]["out"])
    return c_in + c_out

fw_cost = total_cost(fw_tin, fw_tout, "fireworks_llm")
oa_cost = total_cost(oa_tin, oa_tout, "openai_llm")

n = len(golden_set)
print(f"Fireworks: {fw_tin+fw_tout} tokens, ${fw_cost:.6f} total, ${fw_cost/n:.6f}/query")
print(f"OpenAI:    {oa_tin+oa_tout} tokens, ${oa_cost:.6f} total, ${oa_cost/n:.6f}/query")

# Project to scale
for volume in [1_000, 100_000, 1_000_000]:
    print(f"At {volume:>9,} queries:  Fireworks ${fw_cost/n*volume:,.2f}   OpenAI ${oa_cost/n*volume:,.2f}")

### Side-by-side summary + save results

In [ ]:
import json

# fw_scores / oa_scores are already {metric: average} dicts from Stage 3.
summary = {
    "fireworks": {"scores": fw_scores, "cost_per_query_usd": fw_cost / n, "tokens": fw_tin + fw_tout},
    "openai":    {"scores": oa_scores, "cost_per_query_usd": oa_cost / n, "tokens": oa_tin + oa_tout},
}
print(json.dumps(summary, indent=2, default=str))

with open("s10_rag_comparison.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("\nSaved s10_rag_comparison.json")